# XGBoost: признаки stepa330, девять стартов, Optuna
Настройки — в первой кодовой ячейке. Восемь отдельных direct-моделей, пять попыток на горизонт: одна заданная пользователем и четыре Optuna. Базовые признаки победившего режима stepa330 дополнены profile28, соседними сезонными слотами, нормализованными статусами 1–3, недельными изменениями и сетевой нагрузкой. Существующие входные признаки сохранены.
Последние семь полных дней прогноза — test; предыдущие дни хронологически делятся 80/20 на train/val. На train девять стартов 08:30–12:30, на val/test один старт 10:30 и цели 11:00–14:30. 10:30 — доступная история, а не прогнозируемый горизонт.
Test-метрики всех попыток раскрываются только после фиксации параметров всех горизонтов. Ноутбук не создаёт сабмит и не переобучает модели на val/test.


In [ ]:
from pathlib import Path
import json
import hashlib
import gc
import math
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import optuna
import xgboost as xgb
from IPython.display import display
DATA_DIR = None
CACHE_INPUT = None
OUTPUT_DIR = Path('/kaggle/working/xgb_tuning')
DEVICE = 'cuda'
N_JOBS = 4
SEED = 42
N_TRIALS = 5
TEST_DAYS = 7
TRAIN_FRACTION = 0.8
TARGET_SCALE = 100000.0
EARLY_STOPPING_ROUNDS = 100
OBJECTIVE = 'reg:absoluteerror'
MANUAL_PARAMS = {'n_estimators': 1500, 'max_depth': 6, 'learning_rate': 0.05, 'min_child_weight': 10.0, 'subsample': 0.85, 'colsample_bytree': 0.85, 'reg_alpha': 0.01, 'reg_lambda': 10.0}
MANUAL_PARAMS_BY_HORIZON = {
    1: {
        'n_estimators': 1500,
        'max_depth': 6,
        'learning_rate': 0.05,
        'min_child_weight': 10.5,
        'subsample': 0.85,
        'colsample_bytree': 0.85,
        'reg_alpha': 0.0105,
        'reg_lambda': 10.5,
    },
    2: {
        'n_estimators': 1500,
        'max_depth': 6,
        'learning_rate': 0.05,
        'min_child_weight': 10.5,
        'subsample': 0.85,
        'colsample_bytree': 0.85,
        'reg_alpha': 0.0105,
        'reg_lambda': 10.5,
    },
    3: {
        'n_estimators': 1500,
        'max_depth': 6,
        'learning_rate': 0.05,
        'min_child_weight': 9.5,
        'subsample': 0.85,
        'colsample_bytree': 0.85,
        'reg_alpha': 0.0095,
        'reg_lambda': 9.5,
    },
    4: {
        'n_estimators': 1500,
        'max_depth': 6,
        'learning_rate': 0.05,
        'min_child_weight': 9.5,
        'subsample': 0.85,
        'colsample_bytree': 0.85,
        'reg_alpha': 0.0095,
        'reg_lambda': 9.5,
    },
    5: {
        'n_estimators': 1500,
        'max_depth': 6,
        'learning_rate': 0.05,
        'min_child_weight': 9.5,
        'subsample': 0.85,
        'colsample_bytree': 0.85,
        'reg_alpha': 0.0095,
        'reg_lambda': 9.5,
    },
    6: {
        'n_estimators': 1500,
        'max_depth': 6,
        'learning_rate': 0.05,
        'min_child_weight': 9.5,
        'subsample': 0.85,
        'colsample_bytree': 0.85,
        'reg_alpha': 0.0095,
        'reg_lambda': 9.5,
    },
    7: {
        'n_estimators': 1500,
        'max_depth': 6,
        'learning_rate': 0.05,
        'min_child_weight': 9.5,
        'subsample': 0.85,
        'colsample_bytree': 0.85,
        'reg_alpha': 0.0095,
        'reg_lambda': 9.5,
    },
    8: {
        'n_estimators': 1500,
        'max_depth': 6,
        'learning_rate': 0.05,
        'min_child_weight': 9.5,
        'subsample': 0.85,
        'colsample_bytree': 0.85,
        'reg_alpha': 0.0095,
        'reg_lambda': 9.5,
    },
}
SEARCH_SPACE_BY_HORIZON = {
    1: {
        'n_estimators': (500, 2500),
        'max_depth': (3, 9),
        'learning_rate': (0.02, 0.15),
        'min_child_weight': (1.05, 105.0),
        'subsample': (0.65, 1.0),
        'colsample_bytree': (0.65, 1.0),
        'reg_alpha': (1.05e-05, 10.5),
        'reg_lambda': (0.105, 105.0),
    },
    2: {
        'n_estimators': (500, 2500),
        'max_depth': (3, 9),
        'learning_rate': (0.02, 0.15),
        'min_child_weight': (1.05, 105.0),
        'subsample': (0.65, 1.0),
        'colsample_bytree': (0.65, 1.0),
        'reg_alpha': (1.05e-05, 10.5),
        'reg_lambda': (0.105, 105.0),
    },
    3: {
        'n_estimators': (500, 2500),
        'max_depth': (3, 9),
        'learning_rate': (0.02, 0.15),
        'min_child_weight': (0.95, 95.0),
        'subsample': (0.65, 1.0),
        'colsample_bytree': (0.65, 1.0),
        'reg_alpha': (9.5e-06, 9.5),
        'reg_lambda': (0.095, 95.0),
    },
    4: {
        'n_estimators': (500, 2500),
        'max_depth': (3, 9),
        'learning_rate': (0.02, 0.15),
        'min_child_weight': (0.95, 95.0),
        'subsample': (0.65, 1.0),
        'colsample_bytree': (0.65, 1.0),
        'reg_alpha': (9.5e-06, 9.5),
        'reg_lambda': (0.095, 95.0),
    },
    5: {
        'n_estimators': (500, 2500),
        'max_depth': (3, 9),
        'learning_rate': (0.02, 0.15),
        'min_child_weight': (0.95, 95.0),
        'subsample': (0.65, 1.0),
        'colsample_bytree': (0.65, 1.0),
        'reg_alpha': (9.5e-06, 9.5),
        'reg_lambda': (0.095, 95.0),
    },
    6: {
        'n_estimators': (500, 2500),
        'max_depth': (3, 9),
        'learning_rate': (0.02, 0.15),
        'min_child_weight': (0.95, 95.0),
        'subsample': (0.65, 1.0),
        'colsample_bytree': (0.65, 1.0),
        'reg_alpha': (9.5e-06, 9.5),
        'reg_lambda': (0.095, 95.0),
    },
    7: {
        'n_estimators': (500, 2500),
        'max_depth': (3, 9),
        'learning_rate': (0.02, 0.15),
        'min_child_weight': (0.95, 95.0),
        'subsample': (0.65, 1.0),
        'colsample_bytree': (0.65, 1.0),
        'reg_alpha': (9.5e-06, 9.5),
        'reg_lambda': (0.095, 95.0),
    },
    8: {
        'n_estimators': (500, 2500),
        'max_depth': (3, 9),
        'learning_rate': (0.02, 0.15),
        'min_child_weight': (0.95, 95.0),
        'subsample': (0.65, 1.0),
        'colsample_bytree': (0.65, 1.0),
        'reg_alpha': (9.5e-06, 9.5),
        'reg_lambda': (0.095, 95.0),
    },
}


In [ ]:
import ctypes
import warnings

if DEVICE != 'cuda':
    raise ValueError('Этот ноутбук допускает обучение только на GPU: DEVICE должен быть cuda')
if not xgb.build_info().get('USE_CUDA', False):
    raise RuntimeError('Установленная сборка XGBoost не поддерживает CUDA. Нужна CUDA-сборка XGBoost в Kaggle')
try:
    cuda_driver = ctypes.CDLL('libcuda.so.1')
    cuda_status = cuda_driver.cuInit(0)
    gpu_count = ctypes.c_int()
    if cuda_status != 0 or cuda_driver.cuDeviceGetCount(ctypes.byref(gpu_count)) != 0 or gpu_count.value < 1:
        raise RuntimeError('CUDA не видит GPU. Включите GPU accelerator в настройках Kaggle Notebook')
except OSError as error:
    raise RuntimeError('Драйвер NVIDIA недоступен. Включите GPU accelerator в Kaggle') from error
warnings.filterwarnings('error', message='(?s).*(No visible GPU|not compiled with CUDA).*', category=UserWarning)
print(f'CUDA готова. GPU: {gpu_count.value}. XGBoost: {xgb.__version__}')


## Кэш
Первый запуск сохраняет `feature_cache/<fingerprint>/history_h1.parquet` … `history_h8.parquet`, manifest и словарь корреляционных лагов. Повторный запуск проверяет отпечаток исходного parquet, реализации признаков и разбиения. Изменение гиперпараметров XGBoost не сбрасывает кэш.
Для новой Kaggle-сессии сохраните Output предыдущей версии как Dataset и подключите его через Add Input. Совместимый manifest ищется автоматически в `/kaggle/input`; при необходимости укажите `CACHE_INPUT` — точную папку с manifest. `/kaggle/working` сам по себе между новыми сессиями не сохраняется. Незавершённый локальный кэш достраивается по горизонтам.


In [ ]:
TARGET = 'target_1h'
STATUSES = [f'status_{i}' for i in range(1, 7)]
STEP = pd.Timedelta(minutes=30)
START_MINUTES = list(range(510, 751, 30))
PIPELINE_VERSION = 'stepa330_extended_xgb_9starts_v2'
LAG_MAX_LOOKBACK_HOURS = 24
LAG_MIN_PAIRS = 336
LAG_MIN_CORRELATION = 0.05
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR = OUTPUT_DIR / 'runs' / pd.Timestamp.now(tz='UTC').strftime('%Y%m%dT%H%M%S%fZ')
RUN_DIR.mkdir(parents=True, exist_ok=False)

def file_hash(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def save_json(path, payload):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + '.partial')
    temporary.write_text(json.dumps(payload, ensure_ascii=False, indent=2, allow_nan=False))
    temporary.replace(path)
if DATA_DIR is None:
    candidates = list(Path('/kaggle/input').rglob('train_solo_track.parquet'))
    if not candidates and Path('data/train_solo_track.parquet').is_file():
        candidates = [Path('data/train_solo_track.parquet')]
    if len(candidates) != 1:
        raise ValueError(f'Укажите DATA_DIR: найдено {len(candidates)} исходных parquet')
    TRAIN_PATH = candidates[0]
else:
    TRAIN_PATH = Path(DATA_DIR) / 'train_solo_track.parquet'
if OBJECTIVE not in ('reg:absoluteerror', 'reg:squarederror'):
    raise ValueError('OBJECTIVE должен быть reg:absoluteerror или reg:squarederror')
if N_TRIALS != 5:
    raise ValueError('В этом эксперименте предусмотрены ровно пять попыток на горизонт')


In [ ]:
FEATURE_CODE = r"""


import pyarrow as pa
import pyarrow.parquet as pq
target = values[TARGET]

def past(array, positions):
    positions = np.asarray(positions, dtype=np.int64)
    valid = (positions >= 0) & (positions < len(array))
    result = array[np.clip(positions, 0, len(array) - 1)].astype(np.float32)
    result[~valid] = np.nan
    return result

def average_available(matrices):
    total = np.zeros_like(matrices[0], dtype=np.float32)
    count = np.zeros_like(total)
    for matrix in matrices:
        valid = np.isfinite(matrix)
        total += np.where(valid, matrix, 0)
        count += valid
    return np.divide(total, count, out=np.full_like(total, np.nan), where=count > 0)

def rolling(array, steps):
    return pd.DataFrame(array).rolling(steps, min_periods=1).mean().to_numpy(dtype=np.float32)
means = {(name, steps): rolling(array, steps) for name, array in values.items() for steps in (4, 12)}
level24 = rolling(target, 48)
positions = np.arange(len(times))
extra_signals = {}
for name in STATUSES[:3]:
    extra_signals[name + '_hourly'] = values[name] + past(values[name], positions - 1)
extra_signals['warehouse_total_load'] = sum((extra_signals[name + '_hourly'] for name in STATUSES[:3]))
extra_signals['warehouse_processing_rate'] = extra_signals['status_3_hourly'] / (extra_signals['status_1_hourly'] + np.float32(1e-05))
for name in STATUSES[3:]:
    extra_signals[name + '_diff_hourly'] = values[name] - past(values[name], positions - 2)
extra_signals['hub_buffer'] = values['status_5'] - values['status_4']
extra_signals['hub_gate_bottleneck'] = extra_signals['status_4_diff_hourly'] - extra_signals['status_6_diff_hourly']
extra_signals['target_acceleration'] = past(target, positions - 1) - past(target, positions - 2)
hub_signal = extra_signals['status_6_diff_hourly']

def common_features(origins):
    common = {}
    for k in (0, 1, 2, 4):
        common[f'target_back_{k}'] = past(target, origins - k)
    for name, array in values.items():
        for steps in (4, 12):
            common[f'{name}_mean_{steps}'] = means[name, steps][origins]
        if name in STATUSES:
            common[f'{name}_latest'] = array[origins]
            if name in STATUSES[:3]:
                common[f'{name}_change_1h'] = array[origins] - past(array, origins - 2)
    for name, array in extra_signals.items():
        common[name] = array[origins]
    for lag in (48, 96):
        common[f'target_1h_lag_{lag}'] = past(target, origins - lag)
    common['warehouse_total_load_lag_48'] = past(extra_signals['warehouse_total_load'], origins - 48)
    common['status_6_diff_hourly_lag_48'] = past(hub_signal, origins - 48)
    common['target_vs_mean_6h'] = target[origins] - means[TARGET, 12][origins]
    recent = means[TARGET, 4][origins]
    usual = average_available([past(means[TARGET, 4], origins - day * 48) for day in range(1, 8)])
    common['morning_level_ratio'] = np.divide(recent, usual, out=np.full_like(recent, np.nan), where=usual > 0)
    return common
CORRELATION_COLUMNS = ['selected_status6_lag_steps', 'selected_status6_correlation', 'selected_status6_pair_count', 'status6_lag_valid', 'lagged_status6_signal', 'lagged_status6_signal_available']
FEATURE_GROUPS = {'calendar': ['hour', 'minute', 'hour_float', 'hour_sin', 'hour_cos', 'dayofweek', 'is_weekend', 'weekday_sin', 'weekday_cos'], 'local_statuses': ['status_1_hourly', 'status_2_hourly', 'status_3_hourly', 'warehouse_total_load', 'warehouse_processing_rate'], 'hub_combinations': ['status_4_diff_hourly', 'status_5_diff_hourly', 'status_6_diff_hourly', 'hub_buffer', 'hub_gate_bottleneck'], 'historical_lags': ['target_1h_lag_48', 'target_1h_lag_96', 'warehouse_total_load_lag_48', 'status_6_diff_hourly_lag_48'], 'target_change': ['target_acceleration'], 'correlation_lag': CORRELATION_COLUMNS, 'pilot_start_time': ['start_sin', 'start_cos'], 'pilot_peaks': ['peak7_mean_max', 'peak7_mean_level', 'peak7_share_-1', 'peak7_share_0', 'peak7_share_1']}

def calculate_dynamic_shoulder(target_values, status6_diff, timestamps, route_ids, *, fit_before, max_lookback_hours=24, min_pairs=336, min_correlation=0.05, route_block_size=128):
    grid = pd.DatetimeIndex(timestamps)
    route_ids = np.asarray(route_ids)
    target_values, status6_diff = (np.asarray(target_values), np.asarray(status6_diff))
    if target_values.shape != status6_diff.shape or target_values.shape != (len(grid), len(route_ids)):
        raise ValueError('Ожидаются два массива [время, маршрут] одной формы')
    if len(grid) == 0 or not grid.equals(pd.date_range(grid[0], periods=len(grid), freq=STEP)):
        raise ValueError('Для выбора лага нужна упорядоченная получасовая сетка без дубликатов')
    if len(np.unique(route_ids)) != len(route_ids):
        raise ValueError('Идентификаторы маршрутов повторяются')
    steps = float(max_lookback_hours) * 2
    if not np.isfinite(steps) or steps < 2 or (not steps.is_integer()):
        raise ValueError('Граница поиска должна задавать целое число получасов, не меньше двух')
    if not isinstance(min_pairs, (int, np.integer)) or min_pairs < 3:
        raise ValueError('min_pairs должен быть целым числом >=3')
    if not 0 <= min_correlation <= 1 or route_block_size < 1:
        raise ValueError('Некорректные параметры выбора корреляции')
    max_lag = int(steps)
    stop = int(grid.searchsorted(pd.Timestamp(fit_before), side='left'))
    anchors = np.arange(max_lag + 2, stop)
    best_corr = np.full(len(route_ids), np.nan)
    best_lag = np.full(len(route_ids), np.nan)
    best_pairs = np.zeros(len(route_ids), dtype=np.int32)
    for start in range(0, len(route_ids), route_block_size):
        finish = min(start + route_block_size, len(route_ids))
        x = target_values[anchors, start:finish].astype(np.float64)
        for lag in range(2, max_lag + 1):
            z = status6_diff[anchors - lag, start:finish].astype(np.float64)
            valid = np.isfinite(x) & np.isfinite(z)
            count = valid.sum(axis=0)
            xx, zz = (np.where(valid, x, 0), np.where(valid, z, 0))
            divisor = np.maximum(count, 1)
            sx, sz = (xx.sum(axis=0), zz.sum(axis=0))
            covariance = (xx * zz).sum(axis=0) - sx * sz / divisor
            vx = np.maximum((xx * xx).sum(axis=0) - sx * sx / divisor, 0)
            vz = np.maximum((zz * zz).sum(axis=0) - sz * sz / divisor, 0)
            denominator = np.sqrt(vx * vz)
            corr = np.divide(covariance, denominator, out=np.full(finish - start, np.nan), where=(count >= min_pairs) & (denominator > 0))
            corr = np.clip(corr, -1, 1)
            previous = best_corr[start:finish]
            better = np.isfinite(corr) & (~np.isfinite(previous) | (corr > previous))
            best_corr[start:finish][better] = corr[better]
            best_lag[start:finish][better] = lag
            best_pairs[start:finish][better] = count[better]
    accepted = np.isfinite(best_corr) & (best_corr > min_correlation)
    return pd.DataFrame({'route_id': route_ids, 'selected_status6_lag_steps': np.where(accepted, best_lag, np.nan), 'selected_status6_correlation': best_corr, 'selected_status6_pair_count': best_pairs, 'status6_lag_valid': accepted.astype(np.int8)})

def add_correlation_features(frame, h, lag_table, chunk_rows=200000):
    if h not in range(1, 9) or chunk_rows < 1:
        raise ValueError('Некорректный горизонт или размер блока')
    by_route = lag_table.set_index('route_id').reindex(routes)
    route_index = pd.Index(routes)
    lag = by_route.selected_status6_lag_steps.to_numpy(dtype=np.float32)
    quality = {name: by_route[name].to_numpy(dtype=np.float32) for name in CORRELATION_COLUMNS[:4]}
    for name in CORRELATION_COLUMNS:
        frame[name] = np.full(len(frame), np.nan, dtype=np.float32)
    column_positions = {name: frame.columns.get_loc(name) for name in CORRELATION_COLUMNS}
    for start in range(0, len(frame), chunk_rows):
        finish = min(start + chunk_rows, len(frame))
        block = frame.iloc[start:finish]
        origin = times.get_indexer(pd.DatetimeIndex(block.cutoff))
        route = route_index.get_indexer(block.route_id)
        if (origin < 0).any() or (route < 0).any():
            raise ValueError('Строка признаков не сопоставлена с историей')
        selected = lag[route]
        source = origin + h - np.nan_to_num(selected, nan=0).astype(np.int64)
        available = np.isfinite(selected) & (selected >= h) & (source >= 0) & (source <= origin) & (source < len(times))
        signal = hub_signal[np.clip(source, 0, len(times) - 1), route].astype(np.float32)
        available &= np.isfinite(signal)
        for name, array in quality.items():
            frame.iloc[start:finish, column_positions[name]] = array[route]
        frame.iloc[start:finish, column_positions['lagged_status6_signal']] = np.where(available, signal, np.nan)
        frame.iloc[start:finish, column_positions['lagged_status6_signal_available']] = available.astype(np.float32)
    return frame

def horizon_dataset(h, origins):
    slot = origins + h
    dates = times[origins] + h * STEP
    frame = pd.DataFrame({'cutoff': times[origins].repeat(len(routes)), 'timestamp': dates.repeat(len(routes)), 'route_id': np.tile(routes, len(origins))})
    for name, matrix in common_features(origins).items():
        frame[name] = matrix.ravel()
    calendar = {'hour': dates.hour, 'minute': dates.minute, 'hour_float': dates.hour + dates.minute / 60.0, 'dayofweek': dates.dayofweek, 'is_weekend': (dates.dayofweek >= 5).astype(np.int8)}
    calendar['hour_sin'] = np.sin(2 * np.pi * calendar['hour_float'] / 24)
    calendar['hour_cos'] = np.cos(2 * np.pi * calendar['hour_float'] / 24)
    calendar['weekday_sin'] = np.sin(2 * np.pi * dates.dayofweek / 7)
    calendar['weekday_cos'] = np.cos(2 * np.pi * dates.dayofweek / 7)
    for name, array in calendar.items():
        frame[name] = np.repeat(array, len(routes))
    frame = add_correlation_features(frame, h, validation_lag_table)
    for days in (1, 2, 7):
        frame[f'slot_days_back_{days}'] = past(target, slot - days * 48).ravel()
    for name, days in [('slot_daily7', range(1, 8)), ('slot_weekly4', (7, 14, 21, 28))]:
        frame[name] = average_available([past(target, slot - d * 48) for d in days]).ravel()
    profile = np.zeros((len(origins), len(routes)))
    counts = np.zeros_like(profile)
    for day in range(1, 29):
        level = past(level24, origins - day * 48)
        adjusted = past(target, slot - day * 48) * level24[origins] / np.maximum(level, 1)
        profile += np.nan_to_num(adjusted)
        counts += np.isfinite(adjusted)
    frame['profile28'] = np.divide(profile, counts, out=np.full_like(profile, np.nan), where=counts > 0).ravel()
    if h == 1:
        lower = np.zeros_like(target[origins])
        upper = past(target, origins - 2)
        for position in (origins - 1, origins):
            total = past(target, position)
            lower, upper = (np.maximum(0, total - upper), np.minimum(total, total - lower))
        seasonal = frame[['slot_daily7', 'slot_weekly4']].mean(axis=1).to_numpy()
        frame['overlap'] = np.where((lower <= upper).ravel(), (lower + upper).ravel() / 2 + seasonal / 2, seasonal)
    else:
        frame['overlap'] = np.nan
    numeric = frame.columns.difference(['cutoff', 'timestamp', 'route_id'])
    return frame.astype({name: 'float32' for name in numeric})

base_horizon_dataset = horizon_dataset
network_values = {name: array.sum(axis=1, dtype=np.float64).astype(np.float32)[:, None] for name, array in values.items() if name in [TARGET, *STATUSES[:3]]}

def ratio_features(current, usual):
    denominator = np.maximum(usual, np.float32(1.0))
    ratio = np.divide(current, denominator, out=np.full_like(current, np.nan), where=np.isfinite(usual))
    return ratio, np.where(np.isfinite(usual), usual < 1.0, np.nan).astype(np.float32)

def horizon_dataset(h, origins):
    frame = base_horizon_dataset(h, origins)
    added = {}
    slot = origins + h
    profile = {}
    for offset in (-2, -1, 0, 1, 2):
        profile[offset] = average_available([past(target, slot + offset - day * 48) for day in range(1, 8)])
        if offset != 0:
            added[f'slot_daily7_neighbor_{offset:+d}'] = profile[offset].ravel()
    added['slot_daily7_change_2h'] = (profile[2] - profile[-2]).ravel()
    stack = np.stack([profile[offset] for offset in (-2, -1, 0, 1, 2)])
    finite = np.isfinite(stack)
    masked = np.where(finite, stack, -np.inf)
    peak = masked.max(axis=0)
    available = finite.any(axis=0)
    added['slot_daily7_peak_value'] = np.where(available, peak, np.nan).ravel()
    added['slot_daily7_peak_offset_minutes'] = np.where(available, (masked.argmax(axis=0) - 2) * 30, np.nan).ravel()
    for name in STATUSES[:3]:
        for suffix, array in [('latest', values[name]), ('mean_4', means[name, 4])]:
            current = array[origins]
            usual = average_available([past(array, origins - day * 48) for day in range(1, 8)])
            ratio, small = ratio_features(current, usual)
            added[f'{name}_{suffix}_vs_usual7'] = (current - usual).ravel()
            added[f'{name}_{suffix}_ratio_usual7'] = ratio.ravel()
            added[f'{name}_{suffix}_usual7_small'] = small.ravel()
    for day in (14, 21, 28):
        added[f'slot_days_back_{day}'] = past(target, slot - day * 48).ravel()
    added['slot_week1_minus_week2'] = (past(target, slot - 7 * 48) - past(target, slot - 14 * 48)).ravel()
    recent = average_available([past(target, slot - day * 48) for day in (1, 2, 3)])
    previous = average_available([past(target, slot - day * 48) for day in (4, 5, 6, 7)])
    added['slot_recent3_minus_previous4'] = (recent - previous).ravel()
    for name, array in network_values.items():
        current = array[origins]
        usual = average_available([past(array, origins - day * 48) for day in range(1, 8)])
        ratio, small = ratio_features(current, usual)
        features = {'latest': current, 'change_1h': current - past(array, origins - 2),
            'vs_usual7': current - usual, 'ratio_usual7': ratio, 'usual7_small': small}
        for suffix, feature in features.items():
            added[f'network_{name}_{suffix}'] = np.broadcast_to(feature, (len(origins), len(routes))).ravel()
        local = values[name][origins]
        share = np.divide(local, current, out=np.full_like(local, np.nan), where=current > 0)
        past_total = past(array, origins - 2)
        old_share = np.divide(past(values[name], origins - 2), past_total, out=np.full_like(local, np.nan), where=past_total > 0)
        added[f'network_{name}_route_share'] = share.ravel()
        added[f'network_{name}_route_share_change_1h'] = (share - old_share).ravel()
    return pd.concat([frame, pd.DataFrame(added, index=frame.index, dtype=np.float32)], axis=1)
"""


In [ ]:
cache_spec = {'version': PIPELINE_VERSION, 'source_sha256': file_hash(TRAIN_PATH), 'feature_code_sha256': hashlib.sha256(FEATURE_CODE.encode()).hexdigest(), 'starts': START_MINUTES, 'test_days': TEST_DAYS, 'train_fraction': TRAIN_FRACTION, 'lag_settings': [LAG_MAX_LOOKBACK_HOURS, LAG_MIN_PAIRS, LAG_MIN_CORRELATION]}
fingerprint = hashlib.sha256(json.dumps(cache_spec, sort_keys=True).encode()).hexdigest()
LOCAL_CACHE = OUTPUT_DIR / 'feature_cache' / fingerprint

def cache_complete(folder):
    folder = Path(folder)
    try:
        metadata = json.loads((folder / 'manifest.json').read_text())
        if metadata['fingerprint'] != fingerprint:
            return False
        for h in range(1, 9):
            path = folder / f'history_h{h}.parquet'
            record = json.loads(path.with_suffix('.json').read_text())
            if record['fingerprint'] != fingerprint or record['sha256'] != file_hash(path):
                return False
            if pq.ParquetFile(path).metadata.num_rows != record['rows']:
                return False
        return True
    except (OSError, ValueError, KeyError):
        return False
CACHE_DIR = None
if CACHE_INPUT is not None:
    if not cache_complete(CACHE_INPUT):
        raise ValueError('CACHE_INPUT несовместим или неполон')
    CACHE_DIR = Path(CACHE_INPUT)
else:
    cache_candidates = [LOCAL_CACHE]
    cache_candidates.extend((p.parent for p in Path('/kaggle/input').rglob('manifest.json')))
    for candidate in cache_candidates:
        if cache_complete(candidate):
            CACHE_DIR = candidate
            break
if CACHE_DIR is not None:
    manifest = json.loads((CACHE_DIR / 'manifest.json').read_text())
    print('Кэш найден, сборка признаков пропущена:', CACHE_DIR)
else:
    CACHE_DIR = LOCAL_CACHE
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    history = pd.read_parquet(TRAIN_PATH, columns=['route_id', 'timestamp', TARGET, *STATUSES])
    history = history.sort_values(['timestamp', 'route_id'])
    times = pd.DatetimeIndex(history.timestamp.unique())
    routes = np.sort(history.route_id.unique())
    if history.isna().any().any() or history.duplicated(['timestamp', 'route_id']).any():
        raise ValueError('Пропуски или дубликаты в исходных данных')
    if not times.equals(pd.date_range(times[0], times[-1], freq=STEP)):
        raise ValueError('Нарушена получасовая сетка')
    if len(history) != len(times) * len(routes):
        raise ValueError('Неполная панель маршрутов')
    values = {name: history[name].to_numpy(dtype=np.float32).reshape(len(times), len(routes)) for name in [TARGET, *STATUSES]}
    if not all((np.isfinite(v).all() for v in values.values())):
        raise ValueError('Неконечные значения в исходных рядах')
    del history
    end = times[-1]
    dates = times[(times.hour == 10) & (times.minute == 30) & (times + 8 * STEP <= end)]
    if len(dates) < TEST_DAYS + 10:
        raise ValueError('Недостаточно полных дней для разбиения')
    before_test = dates[:-TEST_DAYS]
    split_index = int(math.floor(len(before_test) * TRAIN_FRACTION))
    val_start = before_test[split_index].normalize()
    test_start = dates[-TEST_DAYS].normalize()
    last_day = dates[-1].normalize()
    exec(FEATURE_CODE, globals())
    validation_lag_table = calculate_dynamic_shoulder(target, hub_signal, times, routes, fit_before=val_start, max_lookback_hours=LAG_MAX_LOOKBACK_HOURS, min_pairs=LAG_MIN_PAIRS, min_correlation=LAG_MIN_CORRELATION)
    validation_lag_table.to_csv(CACHE_DIR / 'status6_lags_train.csv', index=False)
    manifest = {'fingerprint': fingerprint, 'spec': cache_spec, 'val_start': str(val_start), 'test_start': str(test_start), 'last_day': str(last_day), 'train_days': split_index, 'val_days': len(before_test) - split_index, 'test_days': TEST_DAYS, 'routes': routes.tolist(), 'lag_fit_before': str(val_start), 'source_end': str(end), 'source_notebook': 'notebooks/stepa/stepa330.ipynb'}
    save_json(CACHE_DIR / 'manifest.json', manifest)
    minutes = times.hour * 60 + times.minute
    eligible_day = times.normalize().isin(dates.normalize())
    train_origins = (times < val_start) & np.isin(minutes, START_MINUTES)
    eval_origins = (times >= val_start) & (minutes == 630)
    origins_all = np.flatnonzero(eligible_day & (train_origins | eval_origins))
    for h in range(1, 9):
        path = CACHE_DIR / f'history_h{h}.parquet'
        try:
            record = json.loads(path.with_suffix('.json').read_text())
            ready = record['fingerprint'] == fingerprint and record['sha256'] == file_hash(path)
        except (OSError, ValueError, KeyError):
            ready = False
        if ready:
            print(f'h{h}: готовый горизонт пропущен')
            continue
        origins = origins_all[origins_all + h < len(times)]
        origins = origins[(times[origins] >= val_start) | (times[origins + h] < val_start)]
        temporary = path.with_suffix('.partial')
        writer = None
        try:
            for start in range(0, len(origins), 36):
                block_origins = origins[start:start + 36]
                frame = horizon_dataset(h, block_origins)
                frame['y'] = target[block_origins + h].ravel()
                frame['split'] = np.where(frame.cutoff < val_start, 'train', np.where(frame.cutoff < test_start, 'val', 'test'))
                table = pa.Table.from_pandas(frame, preserve_index=False)
                if writer is None:
                    writer = pq.ParquetWriter(temporary, table.schema, compression='zstd')
                writer.write_table(table)
        finally:
            if writer is not None:
                writer.close()
        temporary.replace(path)
        save_json(path.with_suffix('.json'), {'fingerprint': fingerprint, 'rows': len(origins) * len(routes), 'sha256': file_hash(path)})
        print(f'h{h}: сохранено {len(origins) * len(routes):,} строк')
    for name in ['values', 'means', 'level24', 'extra_signals', 'target', 'hub_signal', 'network_values', 'frame', 'table']:
        globals().pop(name, None)
    gc.collect()
display(pd.DataFrame([{'split': name, 'days': manifest[name + '_days']} for name in ['train', 'val', 'test']]))
print('Начало val:', manifest['val_start'], 'Начало test:', manifest['test_start'])


## Пять попыток на горизонт
В первой ячейке `MANUAL_PARAMS_BY_HORIZON` содержит восемь отдельных конфигураций: ключи 1–8 соответствуют h1–h8. Меняйте параметры в нужном блоке. Они задают первую попытку соответствующего горизонта, даже за пределами SEARCH_SPACE_BY_HORIZON[h]. Пропущенные параметры берутся из общего MANUAL_PARAMS. Следующие четыре конфигурации выбирает TPE. По умолчанию loss — MAE; для сравнения можно задать `OBJECTIVE = 'reg:squarederror'` без пересборки признаков.
Early stopping использует MAE на val. Победитель выбирается только по MAE в исходных единицах таргета на всех строках val соответствующего горизонта. На фиксированной выборке этот критерий эквивалентен минимуму WAPE. Bias и WAPE + |Relative Bias| сохраняются для отчётности и не влияют на выбор. Это восемь независимых поисков, а не совместная оптимизация общей метрики восьми горизонтов. Дополнительно выводится средняя дневная метрика. Все метрики рассчитываются в исходном масштабе после ограничения прогноза снизу нулём.
Во время поиска показываются train/val; test не выводится до фиксации всех победителей. В конце сохраняется таблица train/val/test для каждого обучения и общая таблица выбранных восьми моделей. Test не используется для early stopping, выбора попыток или количества деревьев.


In [ ]:
def metric_row(frame, prediction, split):
    actual = frame.y.to_numpy(dtype=np.float64)
    error = np.asarray(prediction, dtype=np.float64) - actual
    total = actual.sum()
    if total <= 0 or not np.isfinite(error).all():
        raise ValueError('Метрика неопределена: нулевая сумма цели или неконечный прогноз')
    daily = pd.DataFrame({'day': frame.cutoff.dt.normalize(), 'actual': actual, 'error': error, 'absolute': np.abs(error)}).groupby('day').sum()
    daily = daily.loc[daily.actual > 0]
    wape = np.abs(error).sum() / total
    bias = error.sum() / total
    return {'split': split, 'rows': len(frame), 'MAE': float(np.abs(error).mean()), 'WAPE': float(wape), 'Relative Bias': float(bias), '|Relative Bias|': float(abs(bias)), 'score': float(wape + abs(bias)), 'daily_score': float(((daily.absolute + daily.error.abs()) / daily.actual).mean())}

def suggest_params_h1(trial):
    bounds = SEARCH_SPACE_BY_HORIZON[1]
    return {
        'n_estimators': trial.suggest_int('n_estimators', *bounds['n_estimators']),
        'max_depth': trial.suggest_int('max_depth', *bounds['max_depth']),
        'learning_rate': trial.suggest_float('learning_rate', *bounds['learning_rate'], log=True),
        'min_child_weight': trial.suggest_float('min_child_weight', *bounds['min_child_weight'], log=True),
        'subsample': trial.suggest_float('subsample', *bounds['subsample'], log=False),
        'colsample_bytree': trial.suggest_float('colsample_bytree', *bounds['colsample_bytree'], log=False),
        'reg_alpha': trial.suggest_float('reg_alpha', *bounds['reg_alpha'], log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', *bounds['reg_lambda'], log=True),
    }

def suggest_params_h2(trial):
    bounds = SEARCH_SPACE_BY_HORIZON[2]
    return {
        'n_estimators': trial.suggest_int('n_estimators', *bounds['n_estimators']),
        'max_depth': trial.suggest_int('max_depth', *bounds['max_depth']),
        'learning_rate': trial.suggest_float('learning_rate', *bounds['learning_rate'], log=True),
        'min_child_weight': trial.suggest_float('min_child_weight', *bounds['min_child_weight'], log=True),
        'subsample': trial.suggest_float('subsample', *bounds['subsample'], log=False),
        'colsample_bytree': trial.suggest_float('colsample_bytree', *bounds['colsample_bytree'], log=False),
        'reg_alpha': trial.suggest_float('reg_alpha', *bounds['reg_alpha'], log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', *bounds['reg_lambda'], log=True),
    }

def suggest_params_h3(trial):
    bounds = SEARCH_SPACE_BY_HORIZON[3]
    return {
        'n_estimators': trial.suggest_int('n_estimators', *bounds['n_estimators']),
        'max_depth': trial.suggest_int('max_depth', *bounds['max_depth']),
        'learning_rate': trial.suggest_float('learning_rate', *bounds['learning_rate'], log=True),
        'min_child_weight': trial.suggest_float('min_child_weight', *bounds['min_child_weight'], log=True),
        'subsample': trial.suggest_float('subsample', *bounds['subsample'], log=False),
        'colsample_bytree': trial.suggest_float('colsample_bytree', *bounds['colsample_bytree'], log=False),
        'reg_alpha': trial.suggest_float('reg_alpha', *bounds['reg_alpha'], log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', *bounds['reg_lambda'], log=True),
    }

def suggest_params_h4(trial):
    bounds = SEARCH_SPACE_BY_HORIZON[4]
    return {
        'n_estimators': trial.suggest_int('n_estimators', *bounds['n_estimators']),
        'max_depth': trial.suggest_int('max_depth', *bounds['max_depth']),
        'learning_rate': trial.suggest_float('learning_rate', *bounds['learning_rate'], log=True),
        'min_child_weight': trial.suggest_float('min_child_weight', *bounds['min_child_weight'], log=True),
        'subsample': trial.suggest_float('subsample', *bounds['subsample'], log=False),
        'colsample_bytree': trial.suggest_float('colsample_bytree', *bounds['colsample_bytree'], log=False),
        'reg_alpha': trial.suggest_float('reg_alpha', *bounds['reg_alpha'], log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', *bounds['reg_lambda'], log=True),
    }

def suggest_params_h5(trial):
    bounds = SEARCH_SPACE_BY_HORIZON[5]
    return {
        'n_estimators': trial.suggest_int('n_estimators', *bounds['n_estimators']),
        'max_depth': trial.suggest_int('max_depth', *bounds['max_depth']),
        'learning_rate': trial.suggest_float('learning_rate', *bounds['learning_rate'], log=True),
        'min_child_weight': trial.suggest_float('min_child_weight', *bounds['min_child_weight'], log=True),
        'subsample': trial.suggest_float('subsample', *bounds['subsample'], log=False),
        'colsample_bytree': trial.suggest_float('colsample_bytree', *bounds['colsample_bytree'], log=False),
        'reg_alpha': trial.suggest_float('reg_alpha', *bounds['reg_alpha'], log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', *bounds['reg_lambda'], log=True),
    }

def suggest_params_h6(trial):
    bounds = SEARCH_SPACE_BY_HORIZON[6]
    return {
        'n_estimators': trial.suggest_int('n_estimators', *bounds['n_estimators']),
        'max_depth': trial.suggest_int('max_depth', *bounds['max_depth']),
        'learning_rate': trial.suggest_float('learning_rate', *bounds['learning_rate'], log=True),
        'min_child_weight': trial.suggest_float('min_child_weight', *bounds['min_child_weight'], log=True),
        'subsample': trial.suggest_float('subsample', *bounds['subsample'], log=False),
        'colsample_bytree': trial.suggest_float('colsample_bytree', *bounds['colsample_bytree'], log=False),
        'reg_alpha': trial.suggest_float('reg_alpha', *bounds['reg_alpha'], log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', *bounds['reg_lambda'], log=True),
    }

def suggest_params_h7(trial):
    bounds = SEARCH_SPACE_BY_HORIZON[7]
    return {
        'n_estimators': trial.suggest_int('n_estimators', *bounds['n_estimators']),
        'max_depth': trial.suggest_int('max_depth', *bounds['max_depth']),
        'learning_rate': trial.suggest_float('learning_rate', *bounds['learning_rate'], log=True),
        'min_child_weight': trial.suggest_float('min_child_weight', *bounds['min_child_weight'], log=True),
        'subsample': trial.suggest_float('subsample', *bounds['subsample'], log=False),
        'colsample_bytree': trial.suggest_float('colsample_bytree', *bounds['colsample_bytree'], log=False),
        'reg_alpha': trial.suggest_float('reg_alpha', *bounds['reg_alpha'], log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', *bounds['reg_lambda'], log=True),
    }

def suggest_params_h8(trial):
    bounds = SEARCH_SPACE_BY_HORIZON[8]
    return {
        'n_estimators': trial.suggest_int('n_estimators', *bounds['n_estimators']),
        'max_depth': trial.suggest_int('max_depth', *bounds['max_depth']),
        'learning_rate': trial.suggest_float('learning_rate', *bounds['learning_rate'], log=True),
        'min_child_weight': trial.suggest_float('min_child_weight', *bounds['min_child_weight'], log=True),
        'subsample': trial.suggest_float('subsample', *bounds['subsample'], log=False),
        'colsample_bytree': trial.suggest_float('colsample_bytree', *bounds['colsample_bytree'], log=False),
        'reg_alpha': trial.suggest_float('reg_alpha', *bounds['reg_alpha'], log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', *bounds['reg_lambda'], log=True),
    }

SUGGEST_PARAMS_BY_HORIZON = {1: suggest_params_h1, 2: suggest_params_h2, 3: suggest_params_h3, 4: suggest_params_h4, 5: suggest_params_h5, 6: suggest_params_h6, 7: suggest_params_h7, 8: suggest_params_h8}
all_trial_metrics = []
selected_metrics = []
selected_parts = []
winners = {}
route_dtype = pd.CategoricalDtype(categories=manifest['routes'])
for h in range(1, 9):
    data = pd.read_parquet(CACHE_DIR / f'history_h{h}.parquet')
    data['route_id'] = data.route_id.astype(route_dtype)
    columns = data.columns.drop(['cutoff', 'timestamp', 'y', 'overlap', 'split']).tolist()
    parts = {name: data.loc[data.split.eq(name)].copy() for name in ['train', 'val', 'test']}
    del data
    matrices = {name: frame[columns] for name, frame in parts.items()}
    records = {}
    manual = dict(MANUAL_PARAMS)
    manual.update(MANUAL_PARAMS_BY_HORIZON.get(h, MANUAL_PARAMS_BY_HORIZON.get(str(h), {})))
    if set(manual) != set(SEARCH_SPACE_BY_HORIZON[h]):
        raise ValueError('Ключи ручных параметров должны совпадать с SEARCH_SPACE_BY_HORIZON[h]')
    study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=SEED + h, n_startup_trials=1), storage=f"sqlite:///{(RUN_DIR / 'optuna.sqlite3').resolve()}", study_name=f'h{h}')
    study.enqueue_trial(manual)
    best_value = [float('inf')]

    def objective(trial):
        params = SUGGEST_PARAMS_BY_HORIZON[h](trial)
        trial.set_user_attr('effective_params', params)
        model = xgb.XGBRegressor(**params, objective=OBJECTIVE, eval_metric='mae', tree_method='hist', device=DEVICE, enable_categorical=True, max_cat_to_onehot=1, early_stopping_rounds=EARLY_STOPPING_ROUNDS, random_state=SEED, n_jobs=N_JOBS)
        model.fit(matrices['train'], parts['train'].y / TARGET_SCALE, eval_set=[(matrices['val'], parts['val'].y / TARGET_SCALE)], verbose=False)
        actual_device = json.loads(model.get_booster().save_config())['learner']['generic_param']['device']
        if not actual_device.startswith('cuda'):
            raise RuntimeError(f'XGBoost перешёл на {actual_device}; CPU-обучение запрещено')
        rows = []
        predictions = {}
        for split in ['train', 'val', 'test']:
            prediction = np.maximum(0, model.predict(matrices[split]).astype(np.float64) * TARGET_SCALE)
            predictions[split] = prediction
            row = metric_row(parts[split], prediction, split)
            row.update(horizon=h, trial=trial.number, trees=int(model.best_iteration + 1))
            rows.append(row)
        value = next((row['MAE'] for row in rows if row['split'] == 'val'))
        records[trial.number] = rows
        all_trial_metrics.extend(rows)
        trial.set_user_attr('selected_trees', int(model.best_iteration + 1))
        display(pd.DataFrame([row for row in rows if row['split'] != 'test']))
        if value < best_value[0]:
            best_value[0] = value
            model.save_model(RUN_DIR / f'best_model_h{h}.ubj')
            for split, prediction in predictions.items():
                output = parts[split][['cutoff', 'timestamp', 'route_id', 'y']].copy()
                output['prediction'] = prediction
                output['horizon'] = h
                output['split'] = split
                output.to_parquet(RUN_DIR / f'best_predictions_h{h}_{split}.parquet', index=False)
        del model, predictions
        gc.collect()
        return value
    study.optimize(objective, n_trials=N_TRIALS)
    best = study.best_trial
    winners[str(h)] = {'trial': best.number, 'params': best.user_attrs['effective_params'], 'selected_trees': best.user_attrs['selected_trees'], 'val_mae': best.value, 'feature_names': columns, 'search_space': SEARCH_SPACE_BY_HORIZON[h]}
    selected_metrics.extend(records[best.number])
    study.trials_dataframe().to_csv(RUN_DIR / f'trials_h{h}.csv', index=False)
    save_json(RUN_DIR / f'best_h{h}.json', winners[str(h)])
    for split in ['train', 'val', 'test']:
        selected_parts.append(pd.read_parquet(RUN_DIR / f'best_predictions_h{h}_{split}.parquet'))
    del matrices, parts, records
    gc.collect()
payload = {'feature_fingerprint': fingerprint, 'split': manifest, 'objective': OBJECTIVE, 'target_scale': TARGET_SCALE, 'seed': SEED, 'device': DEVICE, 'xgboost_version': xgb.__version__, 'optuna_version': optuna.__version__, 'n_trials_per_horizon': N_TRIALS, 'search_space_by_horizon': SEARCH_SPACE_BY_HORIZON, 'early_stopping_rounds': EARLY_STOPPING_ROUNDS, 'fixed_model_params': {'tree_method': 'hist', 'enable_categorical': True, 'max_cat_to_onehot': 1, 'eval_metric': 'mae', 'n_jobs': N_JOBS}, 'selection_metric': 'MAE in original target units per horizon on val', 'test_used_for_selection': False, 'full_history_refit': False, 'horizons': winners}
save_json(RUN_DIR / 'best_params.json', payload)
print('Параметры всех горизонтов зафиксированы. Открываем test-метрики.')
trial_table = pd.DataFrame(all_trial_metrics)
trial_table.to_csv(RUN_DIR / 'all_trial_metrics.csv', index=False)
display(trial_table)
selected_table = pd.DataFrame(selected_metrics)
selected_table.to_csv(RUN_DIR / 'selected_metrics_by_horizon.csv', index=False)
display(selected_table)
combined = pd.concat(selected_parts, ignore_index=True)
pooled_rows = []
for split in ['train', 'val', 'test']:
    part = combined.loc[combined.split.eq(split)]
    pooled_rows.append(metric_row(part, part.prediction, split))
pooled = pd.DataFrame(pooled_rows)
pooled.to_csv(RUN_DIR / 'selected_metrics.csv', index=False)
display(pooled)
print('Оценка test получена на моделях, обученных только на train, до переобучения на всей истории.')
print('Эти даты ранее изучались в проекте и не являются новым слепым holdout.')
print('Параметры:', RUN_DIR / 'best_params.json')
print('Кэш для переноса в Kaggle Dataset:', CACHE_DIR)
